# Create La Marató Awards (Fundació La Marató de TV3)

Creates Fundació La Marató de TV3 awards from the foundation's own public
list of funded projects, one page per telethon edition:
https://www.3cat.cat/tv3/marato/en/recerca/projectes-financats/
Each edition is dedicated to one disease area; the projects are selected by
AQuAS-managed international peer review and funded by the foundation. Every
edition page carries a structured JSON module (title, coordinator + partner
groups with institutions, amount in EUR). **1,085 projects, editions
1993-2024, 100% title/PI, 99.9% amount, EUR 224.5M total** (local run
2026-10-01). Edition 1992 funded awareness only (no projects).

**Prerequisites:**
- Run `scripts/local/marato_tv3_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/marato_tv3/marato_tv3_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** the foundation publishes no project code.
Citing works quote its internal expedient codes (e.g. `201331-30` = edition
2013, application 31, partner group 30; ~1,100 `crossref_work_funders` shells
in many spellings), and these cannot be derived from the public list (numbers
follow application order, not the published order). So the key is a stable
synthetic one: `LMTV3-{edition}-{lead-PI-slug}` (a page-sequence suffix only
when one PI leads two projects in an edition: 6 rows). It collapses onto 0
existing citation shells.

**Dates:** only the edition (call) year is published, so start_date/end_date
are NULL and start_year = edition year (PRIN precedent). The edition is held in
December; grants are awarded the following year.

**Scope:** every listed project is ingested (all are research grants). One 2019
project is listed without an amount (amount NULL). Amounts appear in mixed
formats (European `445.312,50 €`, English `€ 300,000.00` in 2018, a few typos);
the script's `parse_amount` normalises them, and per-edition sums match the
totals printed on the edition pages.

**PI names:** split in the script with an Iberian two-surname rule (family =
last two surname units, particles like `de`, `del`, `i` kept with the surname);
see `split_name` in the script.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320310701`
- display_name: Fundació la Marató de TV3
- ror_id / doi: from `openalex.funders.funders` (ror 00t5xc355, doi 10.13039/100008666)

**Priority:** `559` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.marato_tv3_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/marato_tv3/marato_tv3_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.marato_tv3_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.marato_tv3_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.marato_tv3_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320310701 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320310701;

## Step 2: Create La Marató Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.marato_tv3_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320310701  -- Fundació la Marató de TV3
),

parsed AS (
    SELECT
        g.*,
        from_json(g.people, 'ARRAY<STRUCT<name:STRING, given_name:STRING, family_name:STRING, institution:STRING>>') AS people_arr
    FROM openalex.awards.marato_tv3_raw g
    WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
),

people AS (
    -- Coordinator first, then partner-group leaders, in page order.
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(COALESCE(people_arr, ARRAY()), p -> named_struct(
            'given_name', NULLIF(TRIM(p.given_name), ''),
            'family_name', NULLIF(TRIM(p.family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(p.institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        )) AS investigators
    FROM parsed
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CONCAT('La Marató ', g.edition_year, COALESCE(CONCAT(': ', NULLIF(TRIM(g.disease_area), '')), '')) as funder_scheme,
    'marato_tv3_funded_projects' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.edition_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- try_element_at: safe on empty arrays (§2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM parsed g
JOIN people p ON p.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'marato_tv3_funded_projects' AND priority = 559;

-- Insert into openalex_awards_raw with priority.
-- Priority 559: direct-from-funder ingest of La Marató's own funded-projects list.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    559 as priority
FROM openalex.awards.marato_tv3_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.marato_tv3_awards;

In [ ]:
%sql
-- §2.1.1 shape check: every key is the synthetic LMTV3 form.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^LMTV3-(199[3-9]|20[0-9]{2})-[a-z0-9-]+$' THEN 1 ELSE 0 END) as good_shape,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^LMTV3-(199[3-9]|20[0-9]{2})-[a-z0-9-]+$' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.marato_tv3_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, size(investigators) as n_inv
FROM openalex.awards.marato_tv3_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.marato_tv3_awards
GROUP BY start_year, funder_scheme ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.marato_tv3_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.marato_tv3_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage (expect 100% title/PI, 99.9% amount, 0% dates by design).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.marato_tv3_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'marato_tv3_funded_projects'
GROUP BY provenance, priority;